In [106]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [107]:
df = pd.read_csv("car_parts_price_prediction_dataset_2021_2025_with_brands (3).csv")
 

df["Date"] = pd.to_datetime(df["Date"])
df

,Date,Part_Name,Brand,Base_Price_LKR,Price_LKR
0,2021-01-01,EGR Valve,Denso,9500,12700
1,2021-01-01,Wiring Harness,Sumitomo,15000,20200
2,2021-01-02,Wheel Bearing,NTN,10000,14680
3,2021-01-03,CV Axle,Moog,16000,18830
4,2021-01-03,Radiator Hose,Local OEM,2500,1860
...,...,...,...,...,...
5295,2025-06-29,Mass Air Flow Sensor,Bosch,11000,35900
5296,2025-06-29,Radiator Hose,Gates,2500,6040
5297,2025-06-29,Tensioner Pulley,SKF,3200,8290
5298,2025-06-29,Transmission Fluid Filter,Local OEM,4500,7390


In [108]:
print("Shape  :", df.shape)
print("Parts  :", df["Part_Name"].nunique())
print("Brands :", df["Brand"].nunique())

Shape  : (5300, 5)
Parts  : 50
Brands : 67


In [109]:
df["part_brand_key"] = df["Part_Name"].str.strip() + " || " + df["Brand"].str.strip()


In [110]:
le_part_brand = LabelEncoder()
df["part_brand_id"] = le_part_brand.fit_transform(df["part_brand_key"])

In [111]:
le_part  = LabelEncoder().fit(df["Part_Name"])
le_brand = LabelEncoder().fit(df["Brand"])

In [112]:
print("Unique (Part + Brand) combos :", df["part_brand_key"].nunique())

Unique (Part + Brand) combos : 269


In [113]:
price_scaler = MinMaxScaler()
time_scaler  = MinMaxScaler()


In [114]:
df["price_scaled"] = price_scaler.fit_transform(df[["Price_LKR"]])
df["time_idx"]     = (df["Date"] - df["Date"].min()).dt.days
df["time_scaled"]  = time_scaler.fit_transform(df[["time_idx"]])


In [115]:
df = df.sort_values(["part_brand_id", "Date"]).reset_index(drop=True)


In [116]:
def create_sequences(data, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])
        y.append(data[i + window, 0])  # price_scaled
    return np.array(X), np.array(y)

In [117]:
TIME_STEPS = 10
X_all, y_all, skipped = [], [], 0

In [118]:
for combo_id in df["part_brand_id"].unique():
    combo_df = df[df["part_brand_id"] == combo_id].copy()
    if len(combo_df) <= TIME_STEPS:
        skipped += 1
        continue
    # Only 3 features — matching exactly what user provides
    features = combo_df[[
        "price_scaled",
        "part_brand_id",
        "time_scaled",
    ]].values
    X, y = create_sequences(features, TIME_STEPS)
    if len(X) > 0:
        X_all.append(X)
        y_all.append(y)

In [119]:
X = np.vstack(X_all)
y = np.hstack(y_all)
print(f"X shape : {X.shape}")   # (sequences, 10, 3)
print(f"y shape : {y.shape}")

# ── 5. Train / test split ─────────────────────────────────────
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

X shape : (2631, 10, 3)
y shape : (2631,)
Train: 2104  Test: 527


In [120]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(TIME_STEPS, X.shape[2])),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_15 (LSTM)                       │ (None, 10, 128)             │          67,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 10, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_16 (LSTM)                       │ (None, 10, 64)              │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 10, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_17 (LSTM)                       │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 129,953 (507.63 KB)

 Trainable params: 129,953 (507.63 KB)

 Non-trainable params: 0 (0.00 B)

In [121]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
]
model.fit(X_train, y_train, epochs=60, batch_size=32,
          validation_data=(X_test, y_test),
          callbacks=callbacks, verbose=1)


Epoch 1/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - loss: 0.0265 - mae: 0.1256 - val_loss: 0.0128 - val_mae: 0.0968 - learning_rate: 0.0010
Epoch 2/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.0254 - mae: 0.1223 - val_loss: 0.0121 - val_mae: 0.0946 - learning_rate: 0.0010
Epoch 3/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0234 - mae: 0.1171 - val_loss: 0.0182 - val_mae: 0.1132 - learning_rate: 0.0010
Epoch 4/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.0193 - mae: 0.1067 - val_loss: 0.0105 - val_mae: 0.0900 - learning_rate: 0.0010
Epoch 5/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0207 - mae: 0.1100 - val_loss: 0.0104 - val_mae: 0.0885 - learning_rate: 0.0010
Epoch 6/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - loss: 0.0166 - mae: 0.1006 - val_loss: 0.0096 - val_mae: 0.0839 - learning_rate: 0.0010
Epoch 7/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0046 - mae: 0.0493 - val_loss: 0.0013 - val_mae: 0.0286 - learning_rate: 0.0010
Epoch 8/60
6

In [122]:
y_pred_lkr = price_scaler.inverse_transform(model.predict(X_test, verbose=0))
y_true_lkr = price_scaler.inverse_transform(y_test.reshape(-1, 1))
rmse = float(np.sqrt(mean_squared_error(y_true_lkr, y_pred_lkr)))
mae  = float(np.mean(np.abs(y_true_lkr - y_pred_lkr)))
print(f"\nTest RMSE : Rs. {rmse:,.2f}")
print(f"Test MAE  : Rs. {mae:,.2f}\n")



Test RMSE : Rs. 3,055.60
Test MAE  : Rs. 2,401.28



In [124]:
joblib.dump({
    "model"          : model,
    "price_scaler"   : price_scaler,
    "time_scaler"    : time_scaler,
    "le_part_brand"  : le_part_brand,
    "le_part"        : le_part,
    "le_brand"       : le_brand,
    # Dataset reference — needed to build history window at inference
    "dates"          : df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "prices"         : df["Price_LKR"].tolist(),
    "part_brand_keys": df["part_brand_key"].tolist(),
    "partnames"      : df["Part_Name"].tolist(),
    "brands"         : df["Brand"].tolist(),
    "part_brand_ids" : df["part_brand_id"].tolist(),
    "time_steps"     : TIME_STEPS,
    "rmse"           : rmse,
    "mae"            : mae,
}, "price_variation_model_FINAL2.pkl")

print("Saved -> price_variation_model_FINAL2.pkl")
print(f"RMSE : Rs. {rmse:,.2f}  |  MAE : Rs. {mae:,.2f}")

Saved -> price_variation_model_FINAL2.pkl
RMSE : Rs. 3,055.60  |  MAE : Rs. 2,401.28


In [123]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred_lkr = price_scaler.inverse_transform(model.predict(X_test, verbose=0))
y_true_lkr = price_scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = float(np.sqrt(mean_squared_error(y_true_lkr, y_pred_lkr)))
mae  = float(np.mean(np.abs(y_true_lkr - y_pred_lkr)))
r2   = float(r2_score(y_true_lkr, y_pred_lkr))
mape = float(np.mean(np.abs((y_true_lkr - y_pred_lkr) / y_true_lkr)) * 100)

print("=" * 40)
print("  TRAINING ACCURACY REPORT")
print("=" * 40)
print(f"  RMSE     : Rs. {rmse:,.2f}")
print(f"  MAE      : Rs. {mae:,.2f}")
print(f"  R²       :     {r2:.4f}")
print(f"  MAPE     :     {mape:.2f}%")
print("=" * 40)

  TRAINING ACCURACY REPORT
  RMSE     : Rs. 3,055.60
  MAE      : Rs. 2,401.28
  R²       :     0.9274
  MAPE     :     30.34%
